This script creates a complete, runnable Jupyter notebook that demonstrates; how to run media optimization & simulation scenarios for marketing allocation. It also saves a lightweight CSV input template you can replace with your own data.

In [2]:
# -----------------------------
# Create a synthetic CSV input
# -----------------------------
channels = ["paid_search","display","affiliate","social","email"]
weeks = list(range(1, 14))  # one quarter ~13 weeks

In [3]:
import pandas as pd

rows = []
for w in weeks:
    for ch in channels:
        base_impr = {
            "paid_search": 8_000_000,
            "display": 12_000_000,
            "affiliate": 5_000_000,
            "social": 9_000_000,
            "email": 2_500_000,
        }[ch]
        cpi = {
            "paid_search": 0.012,
            "display": 0.004,
            "affiliate": 0.006,
            "social": 0.003,
            "email": 0.0015,
        }[ch]
        freq_cap = {
            "paid_search": 4,
            "display": 5,
            "affiliate": 4,
            "social": 2,
            "email": 6,
        }[ch]
        # CTR and CVR assumptions (toy; will be used inside notebook)
        ctr = {
            "paid_search": 0.035,
            "display": 0.004,
            "affiliate": 0.018,
            "social": 0.008,
            "email": 0.05,
        }[ch]
        cvr = {
            "paid_search": 0.055,
            "display": 0.012,
            "affiliate": 0.030,
            "social": 0.016,
            "email": 0.040,
        }[ch]
        rows.append({
            "week": w,
            "channel": ch,
            "base_impressions": int(base_impr * (0.85 + 0.3 * (w % 3 == 0))), # mild seasonality
            "cpi": cpi,
            "freq_cap_per_user": freq_cap,
            "ctr": ctr,
            "cvr": cvr,
            "avg_order_value": 95.0,             # AOV constant for simplicity
            "existing_users": 250_000           # user pool per week (toy)
        })

df = pd.DataFrame(rows)

df.to_csv(f"media_measurements.csv", index=False)


In [4]:
df.head()

,week,channel,base_impressions,cpi,freq_cap_per_user,ctr,cvr,avg_order_value,existing_users
0,1,paid_search,6800000,0.0120,4,0.035,0.055,95.0,250000
1,1,display,10200000,0.0040,5,0.004,0.012,95.0,250000
2,1,affiliate,4250000,0.0060,4,0.018,0.030,95.0,250000
3,1,social,7650000,0.0030,2,0.008,0.016,95.0,250000
4,1,email,2125000,0.0015,6,0.050,0.040,95.0,250000


In [5]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

# Settings for reproducibility
np.random.seed(42)

# Load inputs (replace with your file)
csv_path = "media_measurements.csv"
try:
    # Try to load your path first
    df = pd.read_csv(csv_path)
except Exception:
    # Fall back to bundled sample
    import os, glob
    candidates = sorted(glob.glob('/data/sample_media_inputs_*.csv'))
    df = pd.read_csv(candidates[-1])
    
df.head()

,week,channel,base_impressions,cpi,freq_cap_per_user,ctr,cvr,avg_order_value,existing_users
0,1,paid_search,6800000,0.0120,4,0.035,0.055,95.0,250000
1,1,display,10200000,0.0040,5,0.004,0.012,95.0,250000
2,1,affiliate,4250000,0.0060,4,0.018,0.030,95.0,250000
3,1,social,7650000,0.0030,2,0.008,0.016,95.0,250000
4,1,email,2125000,0.0015,6,0.050,0.040,95.0,250000


# Media Optimization & Simulation Research Project

**Objective:** Provide a practical, reproducible framework to **run optimization and simulation scenarios** that inform:
- Quarterly **marketing investment & allocation** recommendations
- **Media plan** inputs, **financial forecasting**, and **efficiency gains**
- Insights on **Cost to Acquire (CAC)**, **value of digital engagement**, and **cross-channel impact**

> Replace the sample CSV with your data to run this end-to-end. The notebook includes: data ingestion, adstock & response curves, greedy optimization with budget & frequency caps, quarterly scenarios, Monte Carlo sensitivity, and reporting.

In [6]:
# Create a comprehensive input dataset for the Media Optimization & Simulation Research Project.
# It includes Canadian regions, retail product categories, media channels, weekly baselines,
# cost & performance rates, frequency caps, adstock, and saturation parameters (Hill curve).
#
# It also creates a separate halo matrix CSV (cross-channel spillovers).

import pandas as pd
import numpy as np
from itertools import product
from datetime import date, timedelta

# -------------------------
# Dimensions & assumptions
# -------------------------
regions = [
    "Ontario",
    "Quebec",
    "British Columbia",
    "Alberta",
    "Prairies",
    "Atlantic"
]

product_categories = [
    "Electronics",
    "Apparel",
    "Home & Garden",
    "Grocery",
    "Beauty",
    "Sporting Goods"
]

channels = [
    "Paid Search",
    "Display",
    "Social",
    "Affiliate",
    "Email",
    "Online Video"
]

In [7]:
weeks = list(range(1, 14))  # 13 weeks ~ a quarter
start_date = date(2025, 6, 2)  # Monday start for the quarter example
week_starts = [start_date + timedelta(weeks=w-1) for w in weeks]

In [10]:
# Category-level AOV (CAD) & engagement value per click (EVC) assumptions

category_params = {
    "Electronics":     {"aov": 250.0, "evc": 0.45},
    "Apparel":         {"aov": 85.0,  "evc": 0.25},
    "Home & Garden":   {"aov": 160.0, "evc": 0.30},
    "Grocery":         {"aov": 45.0,  "evc": 0.15},
    "Beauty":          {"aov": 60.0,  "evc": 0.22},
    "Sporting Goods":  {"aov": 140.0, "evc": 0.28},
}

In [11]:
# Region-level size multipliers to scale impressions/users
region_size = {
    "Ontario": 1.00,
    "Quebec": 0.60,
    "British Columbia": 0.45,
    "Alberta": 0.40,
    "Prairies": 0.30,
    "Atlantic": 0.25,
}

In [12]:
# Saturation (Hill) parameters by channel (will be adjusted by region/category scale)
# y = alpha * (x^beta / (x^beta + k^beta))

saturation_params = {
    "Paid Search":  {"alpha": 1.00, "k_basis": 10_000_000, "beta": 1.10, "curve_basis": "impressions"},
    "Display":      {"alpha": 1.00, "k_basis": 25_000_000, "beta": 1.05, "curve_basis": "impressions"},
    "Social":       {"alpha": 1.00, "k_basis": 18_000_000, "beta": 1.10, "curve_basis": "impressions"},
    "Affiliate":    {"alpha": 1.00, "k_basis": 6_000_000,  "beta": 1.00, "curve_basis": "impressions"},
    "Email":        {"alpha": 1.00, "k_basis": 3_000_000,  "beta": 0.95, "curve_basis": "impressions"},
    "Online Video": {"alpha": 1.00, "k_basis": 14_000_000, "beta": 1.15, "curve_basis": "impressions"},
}

In [13]:
# Add spend-based curve parameters too (optional alternative basis)
spend_saturation_params = {
    "Paid Search":  {"alpha_spend": 1.00, "k_spend": 120_000, "beta_spend": 1.05},
    "Display":      {"alpha_spend": 1.00, "k_spend": 90_000,  "beta_spend": 1.05},
    "Social":       {"alpha_spend": 1.00, "k_spend": 80_000,  "beta_spend": 1.08},
    "Affiliate":    {"alpha_spend": 1.00, "k_spend": 60_000,  "beta_spend": 1.00},
    "Email":        {"alpha_spend": 1.00, "k_spend": 40_000,  "beta_spend": 0.95},
    "Online Video": {"alpha_spend": 1.00, "k_spend": 110_000, "beta_spend": 1.10},
}

In [14]:
# Min/Max spend constraints per channel-week (as a fraction of baseline spend)
min_spend_frac = {
    "Paid Search": 0.50, "Display": 0.40, "Social": 0.40,
    "Affiliate":   0.30, "Email":   0.20, "Online Video": 0.30
}
max_spend_frac = {
    "Paid Search": 1.50, "Display": 1.60, "Social": 1.60,
    "Affiliate":   1.80, "Email":   2.50, "Online Video": 1.70
}

In [8]:
# Channel-level baseline assumptions (CTR, CVR, CPI, adstock decay, default frequency caps)
channel_params = {
    "Paid Search":  {"ctr": 0.035, "cvr": 0.055, "cpi": 0.012, "adstock_decay": 0.30, "freq_cap": 4},
    "Display":      {"ctr": 0.004, "cvr": 0.012, "cpi": 0.004, "adstock_decay": 0.50, "freq_cap": 5},
    "Social":       {"ctr": 0.008, "cvr": 0.016, "cpi": 0.003, "adstock_decay": 0.60, "freq_cap": 2},
    "Affiliate":    {"ctr": 0.018, "cvr": 0.030, "cpi": 0.006, "adstock_decay": 0.40, "freq_cap": 4},
    "Email":        {"ctr": 0.050, "cvr": 0.040, "cpi": 0.0015,"adstock_decay": 0.20, "freq_cap": 6},
    "Online Video": {"ctr": 0.003, "cvr": 0.009, "cpi": 0.007, "adstock_decay": 0.70, "freq_cap": 3},
}

In [15]:
# -------------------------
# Build the master dataset
# -------------------------
rows = []

for (region, category, channel, wk) in product(regions, product_categories, channels, weeks):
    # Baseline scale by region & category
    scale = region_size[region] * (1.0 + (0.10 if category in ["Electronics","Apparel"] else 0.0))
    
    # Start from channel defaults
    p = channel_params[channel]
    ctr = p["ctr"] * (0.9 + 0.2 * np.random.rand())
    cvr = p["cvr"] * (0.9 + 0.2 * np.random.rand())
    cpi = p["cpi"] * (0.9 + 0.2 * np.random.rand())
    ad_decay = p["adstock_decay"]
    freq_cap = p["freq_cap"]
    
    # AOV & Engagement per click
    aov = category_params[category]["aov"] * (0.95 + 0.1 * np.random.rand())
    evc = category_params[category]["evc"] * (0.9 + 0.2 * np.random.rand())
    
    # Baseline impressions and users (with mild week seasonality)
    base_impr_level = {
        "Paid Search": 9_000_000, "Display": 12_000_000, "Social": 10_000_000,
        "Affiliate": 5_500_000, "Email": 3_000_000, "Online Video": 6_500_000
    }[channel]
    seasonal = 0.88 + 0.24 * (1 if wk % 4 == 0 else 0)  # every 4th week bumps
    base_impressions = int(base_impr_level * scale * seasonal * (0.9 + 0.2 * np.random.rand()))
    
    # Users (region-scale × category interest)
    users = int(180_000 * region_size[region] * (0.9 + 0.25 * np.random.rand()))
    
    # Baseline spend & costs
    base_spend = base_impressions * cpi
    
    # Saturation parameters (impressions-basis) adjusted by scale
    s = saturation_params[channel].copy()
    s["k_basis_adj"] = max(1_000_0, int(s["k_basis"] * scale))
    
    # Spend-basis optional
    ss = spend_saturation_params[channel].copy()
    ss["k_spend_adj"] = max(10_000, int(ss["k_spend"] * scale))
    
    # Constraints (min/max spend this week)
    min_spend = min_spend_frac[channel] * base_spend
    max_spend = max_spend_frac[channel] * base_spend
    
    rows.append({
        "week_index": wk,
        "week_start": week_starts[wk-1].isoformat(),
        "region": region,
        "product_category": category,
        "channel": channel,
        "currency": "CAD",
        "existing_users": users,
        "base_impressions": base_impressions,
        "base_spend": round(base_spend, 2),
        "cpi": round(cpi, 6),
        "ctr": round(ctr, 6),
        "cvr": round(cvr, 6),
        "avg_order_value": round(aov, 2),
        "engagement_value_per_click": round(evc, 4),
        "freq_cap_per_user": freq_cap,
        "adstock_decay": ad_decay,
        # Saturation (impressions basis)
        "sat_curve_basis": s["curve_basis"],
        "sat_alpha": s["alpha"],
        "sat_k_half_impr": s["k_basis_adj"],
        "sat_beta": s["beta"],
        # Saturation (spend basis)
        "sat_alpha_spend": ss["alpha_spend"],
        "sat_k_half_spend": ss["k_spend_adj"],
        "sat_beta_spend": ss["beta_spend"],
        # Constraints
        "min_spend_this_week": round(min_spend, 2),
        "max_spend_this_week": round(max_spend, 2),
        # Optional planning weights / priorities
        "priority_weight": round(0.9 + 0.2 * np.random.rand(), 3)
    })

master_df = pd.DataFrame(rows)

In [16]:
# --------------------------------------
# Create a halo/spillover matrix (CSV)
# --------------------------------------
# Small symmetric halo: diagonal=1, off-diagonals 0.00–0.05

rng = np.random.default_rng(7)
C = len(channels)
halo = np.eye(C)
for i in range(C):
    for j in range(C):
        if i != j:
            halo[i, j] = np.round(rng.uniform(0.0, 0.05), 3)

halo_df = pd.DataFrame(halo, columns=channels, index=channels).reset_index().rename(columns={"index":"channel"})


In [18]:
master_df.head()

,week_index,week_start,region,product_category,channel,currency,existing_users,base_impressions,base_spend,cpi,...,sat_curve_basis,sat_alpha,sat_k_half_impr,sat_beta,sat_alpha_spend,sat_k_half_spend,sat_beta_spend,min_spend_this_week,max_spend_this_week,priority_weight
0,1,2025-06-02,Ontario,Electronics,Paid Search,CAD,164613,8112604,101868.23,0.012557,...,impressions,1.0,11000000,1.1,1.0,132000,1.05,50934.11,152802.34,1.073
1,2,2025-06-09,Ontario,Electronics,Paid Search,CAD,170182,8210779,89082.05,0.010849,...,impressions,1.0,11000000,1.1,1.0,132000,1.05,44541.02,133623.07,0.937
2,3,2025-06-16,Ontario,Electronics,Paid Search,CAD,175146,8083854,95685.90,0.011837,...,impressions,1.0,11000000,1.1,1.0,132000,1.05,47842.95,143528.84,0.973
3,4,2025-06-23,Ontario,Electronics,Paid Search,CAD,189339,10082208,113719.41,0.011279,...,impressions,1.0,11000000,1.1,1.0,132000,1.05,56859.71,170579.12,0.934
4,5,2025-06-30,Ontario,Electronics,Paid Search,CAD,192790,8010983,105084.20,0.013118,...,impressions,1.0,11000000,1.1,1.0,132000,1.05,52542.10,157626.31,0.988


In [19]:
pwd()

'/workspaces/data-science-portfolio'

In [ ]:
# --------------------------------------
# Save files
# --------------------------------------

master_path = "/workspaces/data-science-portfolio/data/media_inputs_canada_quarterly_with_saturation.csv"
halo_path = "/workspaces/data-science-portfolio/data/halo_matrix_channels.csv"

master_df.to_csv(master_path, index=False)
halo_df.to_csv(halo_path, index=False)

In [23]:
# Re-run the creation since the previous cell reset the state.
import pandas as pd
import numpy as np
from itertools import product
from datetime import date, timedelta

regions = [
    "Ontario","Quebec","British Columbia","Alberta","Prairies","Atlantic"
]
product_categories = [
    "Electronics","Apparel","Home & Garden","Grocery","Beauty","Sporting Goods"
]
channels = ["Paid Search","Display","Social","Affiliate","Email","Online Video"]
weeks = list(range(1,14))
start_date = date(2025,6,2)
week_starts = [start_date + timedelta(weeks=w-1) for w in weeks]

In [24]:
channel_params = {
    "Paid Search":  {"ctr": 0.035, "cvr": 0.055, "cpi": 0.012, "adstock_decay": 0.30, "freq_cap": 4},
    "Display":      {"ctr": 0.004, "cvr": 0.012, "cpi": 0.004, "adstock_decay": 0.50, "freq_cap": 5},
    "Social":       {"ctr": 0.008, "cvr": 0.016, "cpi": 0.003, "adstock_decay": 0.60, "freq_cap": 2},
    "Affiliate":    {"ctr": 0.018, "cvr": 0.030, "cpi": 0.006, "adstock_decay": 0.40, "freq_cap": 4},
    "Email":        {"ctr": 0.050, "cvr": 0.040, "cpi": 0.0015,"adstock_decay": 0.20, "freq_cap": 6},
    "Online Video": {"ctr": 0.003, "cvr": 0.009, "cpi": 0.007, "adstock_decay": 0.70, "freq_cap": 3},
}

In [25]:
category_params = {
    "Electronics":     {"aov": 250.0, "evc": 0.45},
    "Apparel":         {"aov": 85.0,  "evc": 0.25},
    "Home & Garden":   {"aov": 160.0, "evc": 0.30},
    "Grocery":         {"aov": 45.0,  "evc": 0.15},
    "Beauty":          {"aov": 60.0,  "evc": 0.22},
    "Sporting Goods":  {"aov": 140.0, "evc": 0.28},
}


In [26]:
region_size = {
    "Ontario": 1.00, "Quebec": 0.60, "British Columbia": 0.45,
    "Alberta": 0.40, "Prairies": 0.30, "Atlantic": 0.25,
}

In [27]:
saturation_params = {
    "Paid Search":  {"alpha": 1.00, "k_basis": 10_000_000, "beta": 1.10, "curve_basis": "impressions"},
    "Display":      {"alpha": 1.00, "k_basis": 25_000_000, "beta": 1.05, "curve_basis": "impressions"},
    "Social":       {"alpha": 1.00, "k_basis": 18_000_000, "beta": 1.10, "curve_basis": "impressions"},
    "Affiliate":    {"alpha": 1.00, "k_basis": 6_000_000,  "beta": 1.00, "curve_basis": "impressions"},
    "Email":        {"alpha": 1.00, "k_basis": 3_000_000,  "beta": 0.95, "curve_basis": "impressions"},
    "Online Video": {"alpha": 1.00, "k_basis": 14_000_000, "beta": 1.15, "curve_basis": "impressions"},
}

In [28]:
spend_saturation_params = {
    "Paid Search":  {"alpha_spend": 1.00, "k_spend": 120_000, "beta_spend": 1.05},
    "Display":      {"alpha_spend": 1.00, "k_spend": 90_000,  "beta_spend": 1.05},
    "Social":       {"alpha_spend": 1.00, "k_spend": 80_000,  "beta_spend": 1.08},
    "Affiliate":    {"alpha_spend": 1.00, "k_spend": 60_000,  "beta_spend": 1.00},
    "Email":        {"alpha_spend": 1.00, "k_spend": 40_000,  "beta_spend": 0.95},
    "Online Video": {"alpha_spend": 1.00, "k_spend": 110_000, "beta_spend": 1.10},
}

In [29]:
min_spend_frac = {
    "Paid Search": 0.50, "Display": 0.40, "Social": 0.40,
    "Affiliate":   0.30, "Email":   0.20, "Online Video": 0.30
}
max_spend_frac = {
    "Paid Search": 1.50, "Display": 1.60, "Social": 1.60,
    "Affiliate":   1.80, "Email":   2.50, "Online Video": 1.70
}

In [30]:
rows = []
for region in regions:
    for category in product_categories:
        for channel in channels:
            for wk in weeks:
                scale = region_size[region] * (1.0 + (0.10 if category in ["Electronics","Apparel"] else 0.0))
                p = channel_params[channel]
                ctr = p["ctr"] * (0.9 + 0.2 * np.random.rand())
                cvr = p["cvr"] * (0.9 + 0.2 * np.random.rand())
                cpi = p["cpi"] * (0.9 + 0.2 * np.random.rand())
                ad_decay = p["adstock_decay"]
                freq_cap = p["freq_cap"]
                aov = category_params[category]["aov"] * (0.95 + 0.1 * np.random.rand())
                evc = category_params[category]["evc"] * (0.9 + 0.2 * np.random.rand())
                base_impr_level = {
                    "Paid Search": 9_000_000, "Display": 12_000_000, "Social": 10_000_000,
                    "Affiliate": 5_500_000, "Email": 3_000_000, "Online Video": 6_500_000
                }[channel]
                seasonal = 0.88 + 0.24 * (1 if wk % 4 == 0 else 0)
                base_impressions = int(base_impr_level * scale * seasonal * (0.9 + 0.2 * np.random.rand()))
                users = int(180_000 * region_size[region] * (0.9 + 0.25 * np.random.rand()))
                base_spend = base_impressions * cpi

                s = saturation_params[channel].copy()
                s["k_basis_adj"] = max(100000, int(s["k_basis"] * scale))
                ss = spend_saturation_params[channel].copy()
                ss["k_spend_adj"] = max(10000, int(ss["k_spend"] * scale))

                min_spend = min_spend_frac[channel] * base_spend
                max_spend = max_spend_frac[channel] * base_spend

                rows.append({
                    "week_index": wk,
                    "week_start": week_starts[wk-1].isoformat(),
                    "region": region,
                    "product_category": category,
                    "channel": channel,
                    "currency": "CAD",
                    "existing_users": users,
                    "base_impressions": base_impressions,
                    "base_spend": round(base_spend, 2),
                    "cpi": round(cpi, 6),
                    "ctr": round(ctr, 6),
                    "cvr": round(cvr, 6),
                    "avg_order_value": round(aov, 2),
                    "engagement_value_per_click": round(evc, 4),
                    "freq_cap_per_user": freq_cap,
                    "adstock_decay": ad_decay,
                    "sat_curve_basis": s["curve_basis"],
                    "sat_alpha": s["alpha"],
                    "sat_k_half_impr": s["k_basis_adj"],
                    "sat_beta": s["beta"],
                    "sat_alpha_spend": ss["alpha_spend"],
                    "sat_k_half_spend": ss["k_spend_adj"],
                    "sat_beta_spend": ss["beta_spend"],
                    "min_spend_this_week": round(min_spend, 2),
                    "max_spend_this_week": round(max_spend, 2),
                    "priority_weight": round(0.9 + 0.2 * np.random.rand(), 3)
                })

master_df = pd.DataFrame(rows)

In [31]:
# Halo matrix
rng = np.random.default_rng(7)
C = len(channels)
halo = np.eye(C)
for i in range(C):
    for j in range(C):
        if i != j:
            halo[i, j] = np.round(rng.uniform(0.0, 0.05), 3)
halo_df = pd.DataFrame(halo, columns=channels, index=channels).reset_index().rename(columns={"index":"channel"})

In [34]:
master_path = "/workspaces/data-science-portfolio/data/media_inputs_canada_quarterly_with_saturation.csv"
halo_path = "/workspaces/data-science-portfolio/data/halo_matrix_channels.csv"

master_df.to_csv(master_path, index=False)
halo_df.to_csv(halo_path, index=False)


In [37]:
master_df.head()

,week_index,week_start,region,product_category,channel,currency,existing_users,base_impressions,base_spend,cpi,...,sat_curve_basis,sat_alpha,sat_k_half_impr,sat_beta,sat_alpha_spend,sat_k_half_spend,sat_beta_spend,min_spend_this_week,max_spend_this_week,priority_weight
0,1,2025-06-02,Ontario,Electronics,Paid Search,CAD,187300,8970067,113425.66,0.012645,...,impressions,1.0,11000000,1.1,1.0,132000,1.05,56712.83,170138.48,1.052
1,2,2025-06-09,Ontario,Electronics,Paid Search,CAD,192498,8591195,109607.93,0.012758,...,impressions,1.0,11000000,1.1,1.0,132000,1.05,54803.97,164411.90,0.937
2,3,2025-06-16,Ontario,Electronics,Paid Search,CAD,204342,8973050,114041.91,0.012709,...,impressions,1.0,11000000,1.1,1.0,132000,1.05,57020.95,171062.86,1.043
3,4,2025-06-23,Ontario,Electronics,Paid Search,CAD,181099,11637029,144533.35,0.012420,...,impressions,1.0,11000000,1.1,1.0,132000,1.05,72266.68,216800.03,1.042
4,5,2025-06-30,Ontario,Electronics,Paid Search,CAD,173139,8031656,105099.39,0.013086,...,impressions,1.0,11000000,1.1,1.0,132000,1.05,52549.69,157649.08,1.076


## 2) Helper functions: adstock, saturation (diminishing returns), and KPIs

In [38]:
# Load the master media input file
df = pd.read_csv("/workspaces/data-science-portfolio/data/media_inputs_canada_quarterly_with_saturation.csv")

# Aggregate to channel-level panel for the optimizer
panel = df.groupby("channel").agg(
    cost=("base_spend", "sum"),
    conversions=("base_impressions", lambda x: (x * df.loc[x.index, "ctr"] * df.loc[x.index, "cvr"]).sum()),
    revenue=("base_impressions", lambda x: (x * df.loc[x.index, "ctr"] * df.loc[x.index, "cvr"] * df.loc[x.index, "avg_order_value"]).sum())
).reset_index()

panel["roi"] = panel["revenue"] / panel["cost"]

panel.head()

,channel,cost,conversions,revenue,roi
0,Affiliate,7447103.04,6.723255e+05,8.383520e+07,11.257424
1,Display,10863801.82,1.305540e+05,1.622411e+07,1.493410
2,Email,1018390.33,1.367062e+06,1.706298e+08,167.548537
3,Online Video,10247127.40,3.944118e+04,4.895708e+06,0.477764
4,Paid Search,24420602.43,3.894515e+06,4.881368e+08,19.988730


In [39]:
import pandas as pd

def greedy_optimizer(panel, budget, objective="roi", freq_cap=None):
    """
    Simple greedy optimizer for marketing allocation.
    - panel: DataFrame with channel-level cost, conversions, revenue, ROI info
    - budget: total spend budget to allocate
    - objective: "roi" or "lift" (revenue lift)
    - freq_cap: optional max number of increments per channel
    
    Returns:
        alloc_detail: DataFrame of allocations
        summary: Aggregated totals
    """
    allocs = []
    remaining_budget = budget
    
    # copy to avoid modifying original
    df = panel.copy()
    
    # sort by ROI or revenue per cost
    if objective == "roi":
        df["priority"] = df["roi"]
    elif objective == "lift":
        df["priority"] = df["revenue"] / df["cost"]
    else:
        raise ValueError("objective must be 'roi' or 'lift'")
    
    df = df.sort_values("priority", ascending=False).reset_index(drop=True)
    
    for idx, row in df.iterrows():
        if remaining_budget <= 0:
            break
        
        alloc = min(row["cost"], remaining_budget)
        
        # apply frequency cap (if given)
        if freq_cap is not None:
            alloc = min(alloc, row["cost"] * freq_cap)
        
        conversions = row["conversions"] * (alloc / row["cost"])
        revenue = row["revenue"] * (alloc / row["cost"])
        
        allocs.append({
            "channel": row["channel"],
            "allocated_cost": alloc,
            "conversions": conversions,
            "revenue": revenue,
            "roi": revenue / alloc if alloc > 0 else 0
        })
        
        remaining_budget -= alloc
    
    alloc_detail = pd.DataFrame(allocs)
    
    summary = alloc_detail.agg({
        "allocated_cost": "sum",
        "conversions": "sum",
        "revenue": "sum"
    }).to_dict()
    summary["roi"] = summary["revenue"] / summary["allocated_cost"]
    
    return alloc_detail, summary

In [52]:
panel.head()

,channel,cost,conversions,revenue,roi
0,TV,300000,1200,900000,3.0
1,Social,200000,1500,700000,3.5
2,Search,150000,2000,600000,4.0
3,Display,100000,800,250000,2.5
4,Email,50000,400,100000,2.0


In [53]:
panel = pd.read_csv("/workspaces/data-science-portfolio/data/sample_media_inputs_20250822_040242.csv")

# Ensure 'roi' column exists
panel["roi"] = panel["revenue"] / panel["cost"]

alloc_detail, summary = greedy_optimizer(
    panel, 
    budget=1_000_000,   # planning budget
    objective="roi",    # or "lift"
    freq_cap=5          # optional
)

In [54]:
alloc_detail, summary

(     channel  allocated_cost  conversions   revenue  roi
 0  Affiliate           20000        300.0   80000.0  4.0
 1     Search          150000       2000.0  600000.0  4.0
 2     Social          200000       1500.0  700000.0  3.5
 3         TV          300000       1200.0  900000.0  3.0
 4    Display          100000        800.0  250000.0  2.5
 5      Email           50000        400.0  100000.0  2.0,
 {'allocated_cost': 820000.0,
  'conversions': 6200.0,
  'revenue': 2630000.0,
  'roi': 3.207317073170732})

In [44]:
def apply_allocation_and_score(panel: pd.DataFrame, alloc_detail: pd.DataFrame) -> pd.DataFrame:
    df2 = alloc_detail.copy()
    df2['new_impressions'] = df2['base_impressions'] + df2['extra_impressions']
    
    outs = []
    for ch in channels:
        sub = df2[df2.channel==ch].copy().sort_values('week')
        impr = sub['new_impressions'].to_numpy()
        adx = adstock(impr, decay=0.5)
        sat = hill_saturation(adx, alpha=1.0, half_saturation=np.percentile(adx, 75))
        k = kpis_from_impressions(impr, sub['ctr'].iloc[0], sub['cvr'].iloc[0], sub['avg_order_value'].iloc[0])
        cost = cost_from_impressions(impr, sub['cpi'].iloc[0])
        d = pd.DataFrame({
            'week': sub['week'].values,
            'channel': ch,
            'impressions': impr,
            'conversions': k['conversions'],
            'revenue': k['revenue'],
            'cost': cost
        })
        outs.append(d)
    scored = pd.concat(outs, ignore_index=True)
    return scored


In [55]:
# Ensure baseline is defined and has the required columns
# If you don't have a baseline DataFrame, you can use the original panel as baseline
# baseline = panel.copy()
# scored = apply_allocation_and_score(panel, alloc_detail)

baseline_summary = panel.groupby('channel', as_index=False).agg(cost=('cost','sum'), conv=('conversions','sum'), rev=('revenue','sum'))
scenario_summary = alloc_detail.groupby('channel', as_index=False).agg(cost=('allocated_cost','sum'), conv=('conversions','sum'), rev=('revenue','sum'))

summary = baseline_summary.merge(scenario_summary, on='channel', suffixes=('_base','_new'))
summary['delta_cost'] = summary['cost_new'] - summary['cost_base']
summary['delta_conv'] = summary['conv_new'] - summary['conv_base']
summary['delta_rev'] = summary['rev_new'] - summary['rev_base']
summary['marginal_cac'] = summary['delta_cost'] / summary['delta_conv'].replace(0, np.nan)
summary['marginal_roas'] = summary['delta_rev'] / summary['delta_cost'].replace(0, np.nan)
summary

,channel,cost_base,conv_base,rev_base,cost_new,conv_new,rev_new,delta_cost,delta_conv,delta_rev,marginal_cac,marginal_roas
0,Affiliate,20000,300,80000,20000,300.0,80000.0,0,0.0,0.0,NaN,NaN
1,Display,100000,800,250000,100000,800.0,250000.0,0,0.0,0.0,NaN,NaN
2,Email,50000,400,100000,50000,400.0,100000.0,0,0.0,0.0,NaN,NaN
3,Search,150000,2000,600000,150000,2000.0,600000.0,0,0.0,0.0,NaN,NaN
4,Social,200000,1500,700000,200000,1500.0,700000.0,0,0.0,0.0,NaN,NaN
5,TV,300000,1200,900000,300000,1200.0,900000.0,0,0.0,0.0,NaN,NaN


## 6) Quarterly scenarios & forecasting

In [74]:
alloc_detail.head()

,channel,allocated_cost,conversions,revenue,roi
0,Affiliate,20000,300.0,80000.0,4.0
1,Search,150000,2000.0,600000.0,4.0
2,Social,200000,1500.0,700000.0,3.5
3,TV,300000,1200.0,900000.0,3.0
4,Display,100000,800.0,250000.0,2.5


In [ ]:
budgets = {
    "flat": 0,
    "base+250k": 250_000,
    "base+500k": 500_000,
    "base+1M": 1_000_000
}


In [87]:
det, summary = greedy_optimizer(
                    panel, 
                    budget=1_000_000,     # planning budget
                    objective="roi",    # or "lift"
                    freq_cap=30         # optional
                    )
det, summary

(     channel  allocated_cost  conversions   revenue  roi
 0  Affiliate           20000        300.0   80000.0  4.0
 1     Search          150000       2000.0  600000.0  4.0
 2     Social          200000       1500.0  700000.0  3.5
 3         TV          300000       1200.0  900000.0  3.0
 4    Display          100000        800.0  250000.0  2.5
 5      Email           50000        400.0  100000.0  2.0,
 {'allocated_cost': 820000.0,
  'conversions': 6200.0,
  'revenue': 2630000.0,
  'roi': 3.207317073170732})

In [ ]:
def run_quarterly_scenarios(alloc_detail, budgets):
    results = {}
    for name, b in budgets.items():
        det, summary = greedy_optimizer(
                    alloc_detail, 
                    budget=b,   # planning budget
                    objective="roi",    # or "lift"
                    freq_cap=30          # optional
                    )
        # Aggregate KPIs directly from allocation detail
        s = det.groupby('channel', as_index=False).agg(
            cost=('allocated_cost', 'sum'),
            conv=('conversions', 'sum'),
            rev=('revenue', 'sum')
        )
        results[name] = s
    return results

scenarios = run_quarterly_scenarios(alloc_detail, budgets)

KeyError: "Column(s) ['allocated_cost', 'conversions', 'revenue'] do not exist"

In [80]:
# Compare total KPIs
def kpi_totals(df):
    return pd.Series({"cost": df['cost'].sum(), "conv": df['conv'].sum(), "rev": df['rev'].sum(), "roas": df['rev'].sum()/df['cost'].sum()})

totals = pd.DataFrame({name: kpi_totals(scenarios[name]) for name in scenarios}).T
totals

NameError: name 'scenarios' is not defined

## 7) Sensitivity analysis (Monte Carlo on CPI & elasticities)

In [ ]:
def monte_carlo(panel, budget, n=200, cpi_sd=0.15, ctr_sd=0.20, cvr_sd=0.20):
    roas_list = []
    for _ in range(n):
        noisy = panel.copy()
        for ch in channels:
            mask = noisy.channel==ch
            noisy.loc[mask,'cpi'] *= np.random.lognormal(mean=0, sigma=cpi_sd)
            noisy.loc[mask,'ctr'] *= np.random.lognormal(mean=0, sigma=ctr_sd)
            noisy.loc[mask,'cvr'] *= np.random.lognormal(mean=0, sigma=cvr_sd)
        det, _ = greedy_optimizer(noisy, budget, {ch:30 for ch in channels}, objective='roi')
        sc = apply_allocation_and_score(noisy, det)
        roas_list.append(sc['revenue'].sum()/sc['cost'].sum())
    return np.array(roas_list)

mc = monte_carlo(panel, 500_000, n=200)

import matplotlib.pyplot as plt
plt.figure()
plt.hist(mc, bins=25)
plt.title("Monte Carlo ROAS distribution (budget=500k)")
plt.xlabel("ROAS")
plt.ylabel("Count")
plt.show()

mc.mean(), mc.std()

## 8) Cross-channel impact via halo matrix

In [ ]:
H = halo_spillover_matrix(channels)

def apply_halo(conversions_by_channel: pd.Series, H: np.ndarray, channels: list) -> pd.Series:
    vec = conversions_by_channel.values
    haloed = H.dot(vec)
    return pd.Series(haloed, index=channels)

# Example: halo-adjusted conversions for the 'base+500k' scenario
sc = apply_allocation_and_score(panel, greedy_optimize(panel, 500_000, {ch:30 for ch in channels}, objective='roi')[0])
by_ch = sc.groupby('channel')['conversions'].sum().reindex(channels)
halo_adj = apply_halo(by_ch, H, channels)
pd.DataFrame({"raw_conversions": by_ch, "halo_adjusted_conversions": halo_adj})

## 9) Reporting visuals

In [21]:
# Channel-level marginal ROAS from the allocation in Section 5
plt.figure()
plt.bar(summary['channel'], summary['marginal_roas'])
plt.title("Marginal ROAS by Channel")
plt.xlabel("Channel"); plt.ylabel("Marginal ROAS")
plt.show()

# Scenario total ROAS comparison
plt.figure()
plt.bar(totals.index, totals['roas'])
plt.title("Scenario ROAS (Quarter)")
plt.xlabel("Scenario"); plt.ylabel("ROAS")
plt.show()

NameError: name 'summary' is not defined

<Figure size 640x480 with 0 Axes>

## 10) What to deliver to stakeholders

**Outputs you can export:**
1. **Allocation plan**: extra impressions and spend by channel.
2. **Quarterly scenario table**: cost, conversions, revenue, ROAS for each budget level.
3. **Efficiency insights**: CAC and marginal ROAS deltas by channel.
4. **Risk bands**: Monte Carlo ROAS distribution → show expected range.
5. **Cross-channel impact**: Halo-adjusted conversions highlighting spillovers.

**How to use this notebook with your data**
- Replace the CSV path in Section 1 with your file (same column names).
- Tweak `adstock` decay, `hill_saturation` half-saturation, and the halo matrix.
- Change `objective` to `'lift'` if you want to maximize incremental conversions instead of ROI.
- Adjust `freq_caps` and block size in the greedy optimizer to reflect real delivery/frequency constraints.

In [ ]:
nb = new_notebook(cells=cells, metadata={"kernelspec":{"name":"python3","display_name":"Python 3"}})
nbf.write(nb, f"/mnt/data/{nb_name}")

csv_path = f"/mnt/data/{csv_name}"
nb_path = f"/mnt/data/{nb_name}"

from caas_jupyter_tools import display_dataframe_to_user
display_dataframe_to_user("Sample media inputs (you can download this too)", df.head(20))

(csv_path, nb_path)